In [1]:
import numpy as np
print(np.__version__)  # 应显示 1.x.x，如 1.26.4

1.26.4


In [1]:
import torch
import torch.nn as nn
from sentence_transformers import SentenceTransformer
import os
import chromadb
from pypdf import PdfReader
import numpy
import re


# 初始化向量数据库
chroma_client = chromadb.PersistentClient(path="./rule_vector_db") # 数据存在本地文件夹，重启不丢失
collection = chroma_client.get_or_create_collection(
    name="compliance_rules",
    metadata={"hnsw:space": "cosine"} # 用余弦相似度匹配
)

# 加载中文Embedding模型（第一次运行会自动下载，约100M）
embedding_model = SentenceTransformer("BAAI/bge-small-zh-v1.5")

def load_rules_to_rag(rule_folder: str = "./rules"):
    """
    批量入库规则文件，规则文件按目录存放：./rules/美国/出口管制/2026-01-01芯片禁令.pdf
    """
    for root, _, files in os.walk(rule_folder):
        for file in files:
            if not file.endswith(".pdf"): # 支持txt、word等，自行扩展
                continue
            file_path = os.path.join(root, file)
            # 解析路径元数据：国家、法规类型、生效日期
            path_parts = os.path.normpath(root).split(os.sep)
            country = path_parts[-2] if len(path_parts)>=2 else "未知"
            rule_type = path_parts[-1] if len(path_parts)>=1 else "未知"
            effective_date = file.split("_")[0] if "_" in file else "未知"

            # 读取PDF文本
            reader = PdfReader(file_path)
            full_text = "\n".join([page.extract_text() for page in reader.pages if page.extract_text()])
            # 生成按400字符分割的段落列表（替换原paragraphs）
            paragraphs = split_text_by_char_with_punctuation(full_text, chunk_size=400)

            # 后续逻辑可保留原处理（如合并小段落等）
            current_chunk = ""
            chunks = []
            MIN_LEN = 300  # 原逻辑中的最小长度（可根据需求调整）
            MAX_LEN = 500  # 原逻辑中的最大长度（可根据需求调整）

            for para in paragraphs:
                para = para.strip()
                if not para:
                    continue
                temp_chunk = current_chunk + "\n" + para if current_chunk else para
                temp_len = len(temp_chunk)
    
                if temp_len < MIN_LEN:
                    current_chunk = temp_chunk
                elif MIN_LEN <= temp_len <= MAX_LEN:
                    chunks.append(temp_chunk.strip())
                    current_chunk = ""
                else:
                    # 超过最大长度时，优先按标点分割（原逻辑可保留）
                    # 此处可复用上述分割函数的逻辑，或直接添加当前chunk
                    chunks.append(temp_chunk[:MAX_LEN].strip())
                    current_chunk = temp_chunk[MAX_LEN:]

            # 处理最后剩余的chunk
            if current_chunk.strip():
                chunks.append(current_chunk.strip())

            
            # 生成向量并入库
            for idx, chunk in enumerate(chunks):
                print (chunk)
                print ('现在是序号'+str(idx))
                embedding = embedding_model.encode(chunk).tolist()
                collection.add(
                    ids=[f"{country}_{rule_type}_{file}_{idx}"],
                    embeddings=[embedding],
                    documents=[chunk],
                    metadatas=[{
                        "country": country,
                        "rule_type": rule_type,
                        "effective_date": effective_date,
                        "source_file": file_path,
                        "chunk_idx": idx
                    }]
                )
    print(f"入库完成，当前规则库共{collection.count()}条规则块")

def split_text_by_char_with_punctuation(text, chunk_size=400):
    # 预处理：合并多余空白（换行/空格）为单个空格
    cleaned_text = re.sub(r'\s+', ' ', text).strip()
    chunks = []
    start = 0
    text_len = len(cleaned_text)
    # 中英文结束标点（优先在这些位置分割）
    end_puncts = {'.', '?', '!', ';', '。', '？', '！', '；'}
    
    while start < text_len:
        end = min(start + chunk_size, text_len)
        # 向前寻找最近的标点，避免断句
        while end > start and cleaned_text[end-1] not in end_puncts:
            end -= 1
        # 若未找到标点（如全数字），则强制按chunk_size分割
        if end == start:
            end = min(start + chunk_size, text_len)
        # 添加当前chunk（去除前后空白）
        chunk = cleaned_text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start = end
    return chunks




# 执行入库：把你的规则文件放到./rules目录下，运行一次即可
if __name__ == "__main__":
    load_rules_to_rag()

1 出口许可证管理货物目录（2026 年） 序号 货物种类 海关商品编号 货物名称 单位 1 活牛 0102290000 非改良种用家牛 千克/头 0102390010 非改良种用濒危水牛 千克/头 0102390090 非改良种用其他水牛 千克/头 0102909010 非改良种用濒危野牛 千克/头 0102909090 非改良种用其他牛 千克/头 2 活猪 活 大 猪 0103920010 重量在 50 千克及以上的非改良种用濒危猪 千克/头 0103920090 重量在 50 千克及以上的其他非改良种用猪 千克/头 活 中 猪 0103912010 重量在 10 千克及以上但在 50 千克以下的非改良种用濒危猪 千克/头 0103912090 重量在 10 千克及以上但在 50 千克以下的其他非改良种用猪 千克/头 活 乳 猪 0103911010 重量在 10 千克以下的非改良种用
现在是序号0
濒危猪 千克/头 0103911090 重量在 10 千克以下的其他非改良种用猪 千克/头 3 活鸡 0105941000 重量超过 185 克的改良种用鸡 千克/只 0105949000 重量超过 185 克的其他鸡（改良种用的除外） 千克/只 0105999300 重量超过 185 克的非改良种用珍珠鸡 千克/只 4 牛肉 冰 鲜 牛 肉 0201100010 整头及半头鲜或冷藏的濒危野牛肉 千克 0201100090 其他整头及半头鲜或冷藏的牛肉 千克 0201200010 鲜或冷藏的带骨濒危野牛肉 千克 0201200090 其他鲜或冷藏的带骨牛肉 千克 0201300010 鲜或冷藏的去骨濒危野牛肉 千克 0201300090 其他鲜或冷藏的去骨牛肉 千克 2 序号 货物种类 海关商品编号 货物名称 单位 0206100000 鲜或冷藏的牛杂碎 千克 冻 牛 肉 02021000
现在是序号1
10 冻藏的整头及半头濒危野牛肉 千克 0202100090 其他冻藏的整头及半头牛肉 千克 0202200010 冻藏的带骨濒危野牛肉 千克 0202200090 其他冻藏的带骨牛肉 千克 0202300010 冻藏的去骨濒危野牛肉 千克 0202300090 其他冻藏的去骨牛肉 千克 0206210000 冻牛舌 千克 0206220000 冻牛肝 千克 020

In [16]:
import os
print (os.getcwd())

C:\Users\Administrator\guizeku2


In [2]:
with open('.env', 'w', encoding='utf-8') as f:
    f.write('ARK_API_KEY=ark-08df5d04-28d4-4a31-9b9f-7f3d67dca42c-c3e18\n')

In [21]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()
# 这里用豆包API示例，换成开源本地模型的话改下base_url和api_key即可
llm_client = OpenAI(
    api_key=os.getenv("ARK_API_KEY"),
    base_url="https://ark.cn-beijing.volces.com/api/v3"
)

def rag_rule_match(strategy: dict, market: str) -> tuple[bool, list[str], list[dict]]:
    """
    输入：策略字典、目标市场
    输出：(是否合规, 违规原因列表, 关联规则溯源列表)
    """
    # 1. 生成查询文本：把策略的关键字段拼成自然语言查询
    query = f"""
    市场：{market}
    品类：{strategy.get('category', '')}
    产品描述：{strategy.get('product_desc', '')}
    销售方式：{strategy.get('sale_type', '')}
    包装说明：{strategy.get('package_desc', '')}
    """

    # 2. 从向量库召回Top3最相关的规则
    query_embedding = embedding_model.encode(query).tolist()
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=3,
        where={"country": market} # 先过滤国家，缩小召回范围
    )

    # 没有找到对应市场的规则，直接返回无匹配
    if not results["documents"][0]:
        return True, ["未找到对应市场的规则，默认通过"], []

    # 3. 构造LLM提示词，让LLM做判断
    related_rules = "\n---\n".join([f"规则{i+1}：{doc}" for i, doc in enumerate(results["documents"][0])])
    prompt = f"""
    你是专业的合规审核员，请根据给定的【合规规则】，判断【待审核策略】是否违规。
    要求：
    1. 仅基于给出的规则判断，不要用你自己的知识
    2. 输出严格按照JSON格式，不要有额外解释
    3. 如果违规，列出具体的违规原因和对应的规则编号
    4. 规则中未提及的内容，默认视为合规

    【合规规则】：
    {related_rules}

    【待审核策略】：
    {query}

    输出JSON格式：
    {{
        "is_compliance": 布尔值, true=合规, false=违规
        "violations": 数组，每个元素是违规原因字符串
        "related_rule_indexes": 数组，每个元素是关联的规则编号（数字，比如1、2）
    }}
    """

    # 4. 调用LLM获取结果
    response = llm_client.chat.completions.create(
        model="ep-m-20260511001214-hwdzz", # 换成你自己的豆包模型ID
        messages=[{"role": "user", "content": prompt}],
        temperature=0, # 温度设为0，保证输出稳定
        response_format={"type": "json_object"}
    )
    print (response)
    # 5. 解析结果，关联溯源信息
    import json  # 确保文件顶部已导入 json
    try:
        llm_result = json.loads(response.choices[0].message.content)
    except json.JSONDecodeError as e:
        print(f"JSON解析错误: {e}")
        print(f"原始内容: {response.choices[0].message.content}")
        # 返回默认值
        return True, ["解析LLM响应失败"], []
    
    source_infos = []
    for idx in llm_result.get("related_rule_indexes", []):
        if results["metadatas"] and results["metadatas"][0] and idx <= len(results["metadatas"][0]):
            source_infos.append(results["metadatas"][0][idx-1])

    print(llm_result.get("is_compliance"), llm_result.get("violations", []), source_infos)
    return llm_result.get("is_compliance", True), llm_result.get("violations", []), source_infos




def main():
    """
    主函数：演示如何使用 RAG 合规审核
    """
    # 示例1：一个可能的违规策略
    test_strategy_1 = {
        "category": "牛肉",
        "product_desc": "冷冻牛肉",
        "sale_type": "线上直播销售",
        "package_desc": "塑料包装，内含充电线、说明书"
    }
    
    # 示例2：一个合规的策略
    test_strategy_2 = {
        "category": "家居用品",
        "product_desc": "竹制砧板，食品级材料",
        "sale_type": "电商平台销售",
        "package_desc": "可回收纸盒包装"
    }
    
    # 测试市场
    market = "中国"  # 可以改为 "美国"、"中国" 等
    
    print("="*60)
    print(f"正在审核市场：【{market}】的策略...")
    print("="*60)
    
    # 审核策略1
    print("\n📋 策略1：儿童智能手表")
    is_compliant, violations, sources = rag_rule_match(test_strategy_1, market)
    
    if is_compliant:
        print("✅ 审核结果：合规")
    else:
        print("❌ 审核结果：违规")
        print(f"\n违规原因（共{len(violations)}条）：")
        for i, reason in enumerate(violations, 1):
            print(f"  {i}. {reason}")
        
        if sources:
            print(f"\n📚 相关规则来源：")
            for i, source in enumerate(sources, 1):
                print(f"  {i}. {source.get('rule_name', '未命名规则')} - {source.get('country', market)}")
    
    # 审核策略2
    print("\n" + "="*60)
    print("📋 策略2：竹制砧板")
    is_compliant, violations, sources = rag_rule_match(test_strategy_2, market)
    
    if is_compliant:
        print("✅ 审核结果：合规")
    else:
        print("❌ 审核结果：违规")
        print(f"\n违规原因（共{len(violations)}条）：")
        for i, reason in enumerate(violations, 1):
            print(f"  {i}. {reason}")
        
        if sources:
            print(f"\n📚 相关规则来源：")
            for i, source in enumerate(sources, 1):
                print(f"  {i}. {source.get('rule_name', '未命名规则')} - {source.get('country', market)}")
    
    # 批量审核示例
    print("\n" + "="*60)
    print("📊 批量审核示例")
    print("="*60)
    
    strategies_to_review = [
        {"name": "儿童玩具", "data": {
            "category": "玩具",
            "product_desc": "塑胶玩具，含小零件，适合3岁以下儿童",
            "sale_type": "超市零售",
            "package_desc": "塑料袋包装"
        }},
        {"name": "成人保健品", "data": {
            "category": "保健品",
            "product_desc": "维生素C片，每片含1000mg",
            "sale_type": "跨境电商",
            "package_desc": "玻璃瓶包装"
        }}
    ]
    
    for item in strategies_to_review:
        print(f"\n▶ 审核：【{item['name']}】")
        is_compliant, violations, sources = rag_rule_match(item['data'], market)
        print(f"  结果：{'✅ 合规' if is_compliant else '❌ 违规'}")
        if violations and not is_compliant:
            print(f"  原因：{violations[0][:50]}...")



if __name__ == "__main__":
    if not os.getenv("ARK_API_KEY"):
        print("⚠️  警告：未找到 DEEPSEEK_API_KEY 环境变量")
        print("请在 .env 文件中设置: DEEPSEEK_API_KEY=your_api_key")
        print("或者运行: export DEEPSEEK_API_KEY=your_api_key (Linux/Mac)")
        print("或者运行: set DEEPSEEK_API_KEY=your_api_key (Windows)")
    else:
        main()

正在审核市场：【中国】的策略...

📋 策略1：儿童智能手表
ChatCompletion(id='0217784307460649b45e4f03d1ded75da6145fcabce483844a26d', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='{\n    "is_compliance": true,\n    "violations": [],\n    "related_rule_indexes": []\n}', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning_content='\n用户现在给的，合规规则里，首先看，待审核策略是中国市场，品类牛肉冷冻，线上直播卖，包装有塑料包装带充电线说明书？不对，规则里有没有说这个违规？哦，规则里都是商品编号这些，有没有禁止？不对，规则里没提到什么禁止，哦不对，等下，给定的规则，规则1是列了各种商品，规则2规则3其实是规则1的延续吧？哦不对，看问题，待审核策略是卖冷冻牛肉，规则里有没有说违规？哦，规则里只是列了各种商品的海关编码之类的，没有说冷冻牛肉违规啊？不对，包装里有充电线说明书？哦，产品描述是冷冻牛肉，包装里带充电线说明书？不对，规则里没提这个啊，规则说未提及的默认合规？不对不对，再看要求，规则中未提及的内容默认合规。那现在，待审核的是冷冻牛肉，规则里有冻牛肉的编码，没有说不让卖啊？不对，哪里违规了？哦，等下，我再看。哦，不对，用户给的规则，规则1到3其实是出口许可证的目录？不对，题目是让根据给定规则判断，给定规则里，有没有说冷冻牛肉违规？没有啊，那就是合规？不对，包装里有充电线说明书？规则里没说包装不能有这个啊，规则里没提，所以默认合规？哦对，要求说规则中未提及的内容默认视为合规。那is_compliance是true，violations是空数组，related_rule_indexes是空数组？不对，等下再确认。对，给定的规则都是列了各种商品的海关编号货物名称单位，没有说冷冻牛

In [23]:
gdown "https://drive.google.com/uc?id=1ZOYpTUa82_jCcxIdTmyr0LXQfvaM9vIy" -O all_six_datasets.zip

'gdown' 不是内部或外部命令，也不是可运行的程序
或批处理文件。


In [37]:
import pandas as pd
import comtradeapicall

def query_handicraft_exports(reporter_code, partner_code, commodity_codes, year, use_preview=True, subscription_key=None):
    """
    查询手工艺品出口数据并自动求和
    
    参数:
        reporter_code: 报告国代码 (156=中国)
        partner_code: 贸易伙伴国代码 (398=哈萨克斯坦)
        commodity_codes: HS编码列表 (如 ['63', '58', '57'])
        year: 年份
        use_preview: 是否使用preview API (无需密钥，最多500条/次)
        subscription_key: API密钥 (仅当use_preview=False时需要)
    
    返回:
        DataFrame: 合并后的所有数据
        dict: 各编码的贸易额汇总
    """
    results = []
    summary = {}
    
    print(f"\n{'='*80}")
    print(f"查询 {year} 年中国(代码:{reporter_code}) 出口 哈萨克斯坦(代码:{partner_code})")
    print(f"商品编码范围: {commodity_codes}")
    print(f"{'='*80}")
    
    for idx, code in enumerate(commodity_codes, 1):
        print(f"\n[{idx}/{len(commodity_codes)}] 正在查询编码: {code}")
        
        try:
            if use_preview:
                # 使用预览API（无需密钥）
                df = comtradeapicall.previewFinalData(
                    typeCode='C',
                    freqCode='A',
                    clCode='HS',
                    period=str(year),
                    reporterCode=str(reporter_code),
                    cmdCode=str(code),
                    flowCode='X',           # X=出口
                    partnerCode=str(partner_code),
                    partner2Code=None,
                    customsCode=None,
                    motCode=None,
                    maxRecords=5000,        # 增大到5000条
                    format_output='JSON',
                    aggregateBy=None,
                    breakdownMode='classic',
                    countOnly=None,
                    includeDesc=True
                )
            else:
                # 使用正式API（需要密钥）
                df = comtradeapicall.getFinalData(
                    subscription_key,
                    typeCode='C',
                    freqCode='A',
                    clCode='HS',
                    period=str(year),
                    reporterCode=str(reporter_code),
                    cmdCode=str(code),
                    flowCode='X',
                    partnerCode=str(partner_code),
                    partner2Code=None,
                    customsCode=None,
                    motCode=None,
                    maxRecords=5000,
                    format_output='JSON',
                    aggregateBy=None,
                    breakdownMode='classic',
                    countOnly=None,
                    includeDesc=True
                )
            
            if df is not None and not df.empty:
                # 计算该编码的贸易额
                if 'primaryValue' in df.columns:
                    code_total = df['primaryValue'].sum()
                    summary[str(code)] = code_total
                    print(f"   ✅ 成功！找到 {len(df)} 条记录，贸易额: {code_total:,.0f} 美元")
                    
                    # 记录商品描述（取第一条）
                    if 'cmdDescENG' in df.columns:
                        desc = df['cmdDescENG'].iloc[0] if not df.empty else str(code)
                        summary[f"{code}_desc"] = desc[:80]  # 截取前80字符
                    elif 'cmdDesc' in df.columns:
                        desc = df['cmdDesc'].iloc[0] if not df.empty else str(code)
                        summary[f"{code}_desc"] = desc[:80]
                    else:
                        summary[f"{code}_desc"] = str(code)
                else:
                    print(f"   ⚠️ 有数据但无贸易额字段，可用的列: {df.columns.tolist()}")
                    summary[str(code)] = 0
                    summary[f"{code}_desc"] = str(code)
                
                results.append(df)
            else:
                print(f"   ⚠️ 无数据")
                summary[str(code)] = 0
                summary[f"{code}_desc"] = str(code)
                
        except Exception as e:
            print(f"   ❌ 查询失败: {e}")
            summary[str(code)] = 0
            summary[f"{code}_desc"] = str(code)
    
    # 合并所有数据
    if results:
        combined_df = pd.concat(results, ignore_index=True)
        print(f"\n{'='*80}")
        print(f"📊 查询完成！共获取 {len(combined_df)} 条记录")
        return combined_df, summary
    else:
        print(f"\n⚠️ 未找到任何数据")
        return pd.DataFrame(), summary


def print_summary_report(summary, year):
    """
    打印详细的汇总报告 - 修复numpy类型判断问题
    """
    print(f"\n{'='*80}")
    print(f"📈 {year}年中国出口哈萨克斯坦手工艺品贸易额汇总")
    print(f"{'='*80}")
    
    # 只取贸易额相关的键（排除desc结尾的）
    trade_values = {}
    for k, v in summary.items():
        # 排除_desc结尾的键
        if str(k).endswith('_desc'):
            continue
        
        # 检查是否为数字类型（兼容numpy类型）
        try:
            # 尝试转换为float，如果能转换成功就是数字
            float(v)
            trade_values[k] = v
        except (TypeError, ValueError):
            # 不能转换为数字，跳过
            continue
    
    # 检查是否有数据
    if not trade_values:
        print("\n⚠️ 没有找到任何贸易数据！")
        print(f"可用的编码键: {[k for k in summary.keys() if not str(k).endswith('_desc')]}")
        return
    
    # 创建汇总表
    report_data = []
    total = 0
    
    for code, value in trade_values.items():
        # 获取描述
        desc_key = f"{code}_desc"
        description = summary.get(desc_key, str(code))
        
        # 处理description可能是各种类型的情况
        if isinstance(description, (list, tuple)) and len(description) > 0:
            description = str(description[0])[:80]
        elif not isinstance(description, str):
            description = str(description)[:80]
        
        # 确保value是数字
        try:
            numeric_value = float(value)
        except:
            numeric_value = 0
        
        report_data.append({
            'HS编码': code,
            '商品描述(简要)': description,
            '贸易额(美元)': numeric_value
        })
        total += numeric_value
    
    # 创建DataFrame并排序
    if report_data:
        report_df = pd.DataFrame(report_data)
        report_df = report_df.sort_values('贸易额(美元)', ascending=False)
        
        # 打印表格
        print("\n📊 各品类贸易额明细：")
        print(report_df.to_string(index=False))
        
        print(f"\n{'─'*80}")
        print(f"💰 手工艺品出口总额: {total:,.0f} 美元")
        print(f"💰 约合人民币: {total * 7.2:,.0f} 元 (按汇率1:7.2估算)")
        print(f"{'='*80}")
        
        # 找出有数据的编码
        has_data = [f"{code}({value:,.0f})" for code, value in trade_values.items() if value > 0]
        if has_data:
            print(f"\n✅ 有数据的编码: {', '.join(has_data)}")
    else:
        print("\n⚠️ 无法生成汇总报告（无有效数据）")

def test_simple_query():
    """
    测试简单查询，验证API是否正常工作
    """
    print("\n" + "="*80)
    print("🔧 API连通性测试")
    print("="*80)
    
    try:
        # 方法1：查询中国出口全球的总值（使用partnerCode='0'代表世界）
        print("\n测试1: 查询2022年中国出口全球的总值...")
        test_df = comtradeapicall.previewFinalData(
            typeCode='C', freqCode='A', clCode='HS',
            period='2022', reporterCode='156', cmdCode='TOTAL',
            flowCode='X', partnerCode='0',  # ← 关键修改：'WORLD' → '0'
            partner2Code=None, customsCode=None, motCode=None,
            maxRecords=10, format_output='JSON', includeDesc=True
        )
        if test_df is not None and not test_df.empty:
            total_export = test_df['primaryValue'].iloc[0]
            print(f"✅ 测试1成功！2022年中国出口全球总额: {total_export:,.0f} 美元")
            return True
        else:
            print("⚠️ 测试1返回空数据")
            
        # 方法2：如果方法1不行，尝试不指定partnerCode（默认为全球）
        print("\n测试2: 不指定partnerCode（使用默认值）...")
        test_df2 = comtradeapicall.previewFinalData(
            typeCode='C', freqCode='A', clCode='HS',
            period='2022', reporterCode='156', cmdCode='TOTAL',
            flowCode='X', partnerCode=None,  # 不指定，查询所有伙伴
            partner2Code=None, customsCode=None, motCode=None,
            maxRecords=10, format_output='JSON', includeDesc=True
        )
        if test_df2 is not None and not test_df2.empty:
            total_export2 = test_df2['primaryValue'].sum()
            print(f"✅ 测试2成功！2022年中国对已报告伙伴出口总额: {total_export2:,.0f} 美元")
            return True
            
        print("⚠️ API测试返回空数据")
        return False
        
    except Exception as e:
        print(f"❌ API测试失败: {e}")
        return False

# ==================== 主程序 ====================

# 手工艺品HS编码清单
handicraft_codes = [
    # === 纺织类手工艺品 ===
    '58',           # 特种机织物、花边、装饰毯、刺绣品（核心手工区）
    '63',           # 其他制成纺织制品（含手工编织的靠垫套等）
    '57',           # 地毯及其他纺织铺地制品（含手工打结地毯）

    # === 非纺织类手工艺品 ===
    '46',           # 稻草/柳条等编结材料制品（篮子、席子等）
    '6913',         # 陶瓷塑像及其他装饰品
    '4420',         # 木制小雕像、镶嵌木等装饰品
    '7117',         # 仿首饰（含手工项链/手镯）
]

# 先测试API连通性
api_ok = test_simple_query()



# 先测试API连通性（使用修正后的函数）

if api_ok:
    print("\n" + "="*80)
    print("中国出口哈萨克斯坦手工艺品贸易数据分析系统")
    print("="*80)
    
    # 查询2022年数据 - 注意partnerCode使用正确的值
    df_2022, summary_2022 = query_handicraft_exports(
        reporter_code=156,      # 中国
        partner_code=398,       # 哈萨克斯坦（数字代码有效）
        commodity_codes=handicraft_codes,
        year=2022,
        use_preview=True
    )
    
    # 打印汇总报告
    if not df_2022.empty:
        print_summary_report(summary_2022, 2022)
    else:
        print("\n⚠️ 2022年未找到数据，尝试2021年...")
        df_2021, summary_2021 = query_handicraft_exports(
            reporter_code=156, partner_code=398,
            commodity_codes=handicraft_codes,
            year=2021, use_preview=True
        )
        print_summary_report(summary_2021, 2021)
else:
    print("\n⚠️ API连接失败，请检查：")
    print("1. 网络连接是否正常")
    print("2. 可访问 https://comtradeplus.un.org/ 官网验证数据是否存在")
    print("3. 如需更稳定的访问，可申请免费API密钥")


🔧 API连通性测试

测试1: 查询2022年中国出口全球的总值...
✅ 测试1成功！2022年中国出口全球总额: 3,593,601,435,602 美元

中国出口哈萨克斯坦手工艺品贸易数据分析系统

查询 2022 年中国(代码:156) 出口 哈萨克斯坦(代码:398)
商品编码范围: ['58', '63', '57', '46', '6913', '4420', '7117']

[1/7] 正在查询编码: 58
   ✅ 成功！找到 1 条记录，贸易额: 27,241,030 美元

[2/7] 正在查询编码: 63
   ✅ 成功！找到 1 条记录，贸易额: 273,332,462 美元

[3/7] 正在查询编码: 57
   ✅ 成功！找到 1 条记录，贸易额: 14,648,904 美元

[4/7] 正在查询编码: 46
   ✅ 成功！找到 1 条记录，贸易额: 1,145,150 美元

[5/7] 正在查询编码: 6913
   ✅ 成功！找到 1 条记录，贸易额: 2,524,725 美元

[6/7] 正在查询编码: 4420
   ✅ 成功！找到 1 条记录，贸易额: 397,180 美元

[7/7] 正在查询编码: 7117
   ✅ 成功！找到 1 条记录，贸易额: 44,509,761 美元

📊 查询完成！共获取 7 条记录

📈 2022年中国出口哈萨克斯坦手工艺品贸易额汇总

📊 各品类贸易额明细：
HS编码                                                                         商品描述(简要)     贸易额(美元)
  63  Textiles, made up articles; sets; worn clothing and worn textile articles; rags 273332462.0
7117                                                              Imitation jewellery  44509761.0
  58 Fabrics; special woven fabrics, tufted textile fabrics, lace, t

In [42]:
import sys
import subprocess

print("="*50)
print("当前环境包版本：")
print("="*50)

# 使用 pip 直接查询
packages = ['requests', 'urllib3', 'selenium', 'setuptools']
for package in packages:
    try:
        result = subprocess.run([sys.executable, '-m', 'pip', 'show', package], 
                                capture_output=True, text=True)
        for line in result.stdout.split('\n'):
            if line.startswith('Version:'):
                print(f"{package}: {line.split(':')[1].strip()}")
                break
        else:
            print(f"{package}: 未安装")
    except:
        print(f"{package}: 查询失败")

print("\n" + "="*50)
print("Python 版本：")
print("="*50)
print(sys.version)

print("\n" + "="*50)
print("测试导入：")
print("="*50)

# 测试具体导入
try:
    from selenium.webdriver.support.ui import WebDriverWait
    print("✓ WebDriverWait 导入成功")
except Exception as e:
    print(f"✗ WebDriverWait 导入失败: {e}")

try:
    from selenium.webdriver.remote.webdriver import WebDriver
    print("✓ WebDriver 导入成功")
except Exception as e:
    print(f"✗ WebDriver 导入失败: {e}")

try:
    import urllib3
    print(f"✓ urllib3 版本: {urllib3.__version__}")
    print(f"  urllib3.BaseHTTPResponse 存在: {hasattr(urllib3, 'BaseHTTPResponse')}")
    if hasattr(urllib3, 'BaseHTTPResponse'):
        from urllib3 import BaseHTTPResponse
        print(f"  BaseHTTPResponse 类型: {type(BaseHTTPResponse)}")
except Exception as e:
    print(f"✗ urllib3 检查失败: {e}")

try:
    import requests
    print(f"✓ requests 版本: {requests.__version__}")
except Exception as e:
    print(f"✗ requests 检查失败: {e}")

当前环境包版本：
requests: 2.31.0
urllib3: 2.6.3
selenium: 4.44.0
setuptools: 82.0.1

Python 版本：
3.10.20 | packaged by conda-forge | (main, Mar  5 2026, 16:36:49) [MSC v.1944 64 bit (AMD64)]

测试导入：
✗ WebDriverWait 导入失败: module 'urllib3' has no attribute 'BaseHTTPResponse'
✗ WebDriver 导入失败: module 'urllib3' has no attribute 'BaseHTTPResponse'
✓ urllib3 版本: 1.26.13
  urllib3.BaseHTTPResponse 存在: False
✓ requests 版本: 2.28.1


In [1]:
import urllib3
import requests
from selenium.webdriver.support.ui import WebDriverWait

print(f"urllib3: {urllib3.__version__}")
print(f"requests: {requests.__version__}")
print(f"urllib3.BaseHTTPResponse 存在: {hasattr(urllib3, 'BaseHTTPResponse')}")
print("✓ 所有导入成功")

urllib3: 2.6.3
requests: 2.31.0
urllib3.BaseHTTPResponse 存在: True
✓ 所有导入成功


In [4]:
import numpy as np
import pandas as pd
from collections import defaultdict
from datetime import datetime, timedelta
import random
from typing import List, Dict, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

# ================== 离线数据集构建器 ==================
class OfflineDatasetBuilder:
    """从历史数据构建离线训练数据集"""
    def __init__(self):
        self.dataset = []
        
    def build_from_csv(self, filepath: str) -> pd.DataFrame:
        df = pd.read_csv(filepath)
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        return df
    
    def generate_synthetic_history(self, n_samples: int = 10000) -> pd.DataFrame:
        """生成模拟历史数据"""
        np.random.seed(42)
        
        markets = ['Middle_East', 'Southeast_Asia', 'Europe', 'North_America', 'Latin_America']
        categories = ['decor', 'textile', 'ceramic', 'jewelry', 'furniture']
        
        data = {
            'timestamp': [datetime.now() - timedelta(days=np.random.randint(0, 365)) 
                         for _ in range(n_samples)],
            'market': np.random.choice(markets, n_samples),
            'product_category': np.random.choice(categories, n_samples),
            'price_usd': np.random.uniform(20, 150, n_samples),
            'marketing_budget_pct': np.random.uniform(0, 80, n_samples),
            'logistics_cost': np.random.uniform(100, 600, n_samples),
            'tariff_rate': np.random.uniform(0.05, 0.30, n_samples),
            'brand_power_before': np.random.uniform(0.3, 0.9, n_samples),
            'cultural_distance': np.random.uniform(0.2, 0.8, n_samples),
            'digital_maturity': np.random.uniform(0.3, 0.9, n_samples),
            'logistics_accessibility': np.random.uniform(0.4, 0.95, n_samples),
        }
        
        df = pd.DataFrame(data)
        
        # 模拟转化率
        df['conversion_rate'] = (
            0.3 * (1 - df['price_usd'] / 200) +
            0.2 * (df['marketing_budget_pct'] / 100) +
            0.15 * df['brand_power_before'] +
            0.1 * (1 - df['cultural_distance']) +
            0.1 * (1 / (1 + df['tariff_rate'] * 2)) +
            np.random.normal(0, 0.05, n_samples)
        )
        df['conversion_rate'] = df['conversion_rate'].clip(0.05, 0.95)
        
        # 模拟品牌力变化
        df['brand_power_after'] = df['brand_power_before'] + 0.1 * df['conversion_rate']
        df['brand_power_after'] = df['brand_power_after'].clip(0, 1)
        
        # 模拟奖励分数
        df['reward'] = (
            0.5 * df['conversion_rate'] +
            0.3 * (df['brand_power_after'] - df['brand_power_before']) +
            0.2 * (df['price_usd'] - df['logistics_cost']) / df['price_usd']
        )
        
        return df


# ================== 完全修复的离线强化学习训练器 ==================
class OfflineRLTrainer:
    """使用历史数据进行离线PPO训练（完全修复版）"""
    
    def __init__(self, state_dim: int = 5, action_dim: int = 5, hidden_dim: int = 32):
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.hidden_dim = hidden_dim
        
        # 初始化Actor网络权重
        self.actor_W1 = np.random.randn(state_dim, hidden_dim) * 0.01
        self.actor_b1 = np.zeros((1, hidden_dim))  # 改为2D数组
        self.actor_W2 = np.random.randn(hidden_dim, action_dim) * 0.01
        self.actor_b2 = np.zeros((1, action_dim))  # 改为2D数组
        
        # 初始化Critic网络权重
        self.critic_W1 = np.random.randn(state_dim, hidden_dim) * 0.01
        self.critic_b1 = np.zeros((1, hidden_dim))  # 改为2D数组
        self.critic_W2 = np.random.randn(hidden_dim, 1) * 0.01
        self.critic_b2 = np.zeros((1, 1))  # 改为2D数组
        
        self.learning_rate = 0.001
        self.gamma = 0.95
        
    def _relu(self, x):
        return np.maximum(0, x)
    
    def _softmax(self, x):
        exp_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
        return exp_x / np.sum(exp_x, axis=-1, keepdims=True)
    
    def _forward_actor(self, state):
        """策略网络前向传播"""
        # 确保state是2D数组 (batch_size, state_dim)
        if state.ndim == 1:
            state = state.reshape(1, -1)
        
        # 第一层
        hidden = self._relu(np.dot(state, self.actor_W1) + self.actor_b1)
        # 第二层
        logits = np.dot(hidden, self.actor_W2) + self.actor_b2
        # Softmax
        probs = self._softmax(logits)
        
        return probs, logits, hidden
    
    def _forward_critic(self, state):
        """价值网络前向传播"""
        # 确保state是2D数组 (batch_size, state_dim)
        if state.ndim == 1:
            state = state.reshape(1, -1)
        
        # 第一层
        hidden = self._relu(np.dot(state, self.critic_W1) + self.critic_b1)
        # 第二层
        value = np.dot(hidden, self.critic_W2) + self.critic_b2
        
        return value[0, 0] if value.size == 1 else value, hidden
    
    def extract_state_from_history(self, row: pd.Series) -> np.ndarray:
        """从历史数据提取状态向量"""
        return np.array([
            float(row.get('brand_power_before', 0.5)),
            float(row.get('digital_maturity', 0.5)),
            float(row.get('cultural_distance', 0.5)),
            float(row.get('tariff_rate', 0.15)),
            float(row.get('logistics_accessibility', 0.7))
        ])
    
    def extract_action_from_history(self, row: pd.Series) -> int:
        """从历史数据提取动作"""
        price = row.get('price_usd', 50)
        if price < 30:
            return 0
        elif price < 50:
            return 1
        elif price < 70:
            return 2
        elif price < 100:
            return 3
        else:
            return 4
    
    def extract_reward(self, row: pd.Series) -> float:
        """提取或计算奖励值"""
        if 'reward' in row:
            return float(row['reward'])
        
        conversion = float(row.get('conversion_rate', 0.3))
        price = float(row.get('price_usd', 50))
        logistics = float(row.get('logistics_cost', 30))
        profit_margin = (price - logistics) / price if price > 0 else 0
        return 0.6 * conversion + 0.4 * max(0, profit_margin)
    
    def compute_actor_gradients(self, state, action, advantage):
        """计算Actor网络梯度"""
        # 确保state是2D
        if state.ndim == 1:
            state = state.reshape(1, -1)
        
        # 前向传播
        hidden = self._relu(np.dot(state, self.actor_W1) + self.actor_b1)
        logits = np.dot(hidden, self.actor_W2) + self.actor_b2
        probs = self._softmax(logits)
        
        # 计算策略梯度
        grad_logits = probs.copy()
        grad_logits[0, action] -= 1  # 注意索引
        
        # 第二层梯度
        grad_W2 = np.dot(hidden.T, grad_logits)
        grad_b2 = np.sum(grad_logits, axis=0, keepdims=True)
        
        # 第一层梯度
        grad_hidden = np.dot(grad_logits, self.actor_W2.T)
        grad_hidden = grad_hidden * (hidden > 0).astype(float)
        
        grad_W1 = np.dot(state.T, grad_hidden)
        grad_b1 = np.sum(grad_hidden, axis=0, keepdims=True)
        
        # 应用优势函数
        grad_W1 *= advantage
        grad_b1 *= advantage
        grad_W2 *= advantage
        grad_b2 *= advantage
        
        return {
            'W1': grad_W1,
            'b1': grad_b1,
            'W2': grad_W2,
            'b2': grad_b2
        }
    
    def compute_critic_gradients(self, state, td_error):
        """计算Critic网络梯度"""
        # 确保state是2D
        if state.ndim == 1:
            state = state.reshape(1, -1)
        
        # 前向传播
        hidden = self._relu(np.dot(state, self.critic_W1) + self.critic_b1)
        value = np.dot(hidden, self.critic_W2) + self.critic_b2
        
        # 价值损失梯度
        grad_value = td_error
        
        # 第二层梯度
        grad_W2 = np.dot(hidden.T, grad_value)
        grad_b2 = np.sum(grad_value, axis=0, keepdims=True)
        
        # 第一层梯度
        grad_hidden = np.dot(grad_value, self.critic_W2.T)
        grad_hidden = grad_hidden * (hidden > 0).astype(float)
        
        grad_W1 = np.dot(state.T, grad_hidden)
        grad_b1 = np.sum(grad_hidden, axis=0, keepdims=True)
        
        return {
            'W1': grad_W1,
            'b1': grad_b1,
            'W2': grad_W2,
            'b2': grad_b2
        }
    
    def update_network(self, gradients, network_type='actor'):
        """更新网络权重"""
        lr = self.learning_rate
        
        if network_type == 'actor':
            self.actor_W1 -= lr * gradients['W1']
            self.actor_b1 -= lr * gradients['b1']
            self.actor_W2 -= lr * gradients['W2']
            self.actor_b2 -= lr * gradients['b2']
        else:  # critic
            self.critic_W1 -= lr * gradients['W1']
            self.critic_b1 -= lr * gradients['b1']
            self.critic_W2 -= lr * gradients['W2']
            self.critic_b2 -= lr * gradients['b2']
    
    def train_from_batch(self, historical_df: pd.DataFrame, epochs: int = 30, 
                        batch_size: int = 256, verbose: bool = True):
        """从历史数据批量训练"""
        print(f"\n{'='*60}")
        print(f"离线强化学习训练开始")
        print(f"历史数据量: {len(historical_df)} 条记录")
        print(f"训练轮数: {epochs}, 批次大小: {batch_size}")
        print(f"{'='*60}\n")
        
        loss_history = []
        
        for epoch in range(epochs):
            # 随机采样批次
            batch = historical_df.sample(n=min(batch_size, len(historical_df)))
            
            epoch_actor_losses = []
            epoch_critic_losses = []
            
            for _, row in batch.iterrows():
                # 提取数据
                state = self.extract_state_from_history(row)
                action = self.extract_action_from_history(row)
                reward = self.extract_reward(row)
                
                # 计算当前价值
                current_value, _ = self._forward_critic(state)
                
                # TD误差（确保是标量）
                td_error = reward - current_value
                
                # 计算梯度
                actor_grads = self.compute_actor_gradients(state, action, td_error)
                critic_grads = self.compute_critic_gradients(state, td_error)
                
                # 更新网络
                self.update_network(actor_grads, 'actor')
                self.update_network(critic_grads, 'critic')
                
                # 计算损失
                probs, _, _ = self._forward_actor(state)
                log_prob = np.log(probs[0, action] + 1e-8)
                policy_loss = -log_prob * td_error
                value_loss = td_error ** 2
                
                epoch_actor_losses.append(policy_loss)
                epoch_critic_losses.append(value_loss)
            
            avg_actor_loss = np.mean(epoch_actor_losses)
            avg_critic_loss = np.mean(epoch_critic_losses)
            total_loss = avg_actor_loss + 0.5 * avg_critic_loss
            loss_history.append(total_loss)
            
            if verbose and (epoch + 1) % 5 == 0:  # 每5轮打印一次
                print(f"Epoch {epoch+1}/{epochs} - "
                      f"Actor Loss: {avg_actor_loss:.4f}, "
                      f"Critic Loss: {avg_critic_loss:.4f}, "
                      f"Total: {total_loss:.4f}")
        
        print(f"\n✓ 训练完成！最终损失: {loss_history[-1]:.4f}")
        return loss_history
    
    def evaluate_on_holdout(self, test_df: pd.DataFrame) -> Dict:
        """评估模型性能"""
        predictions = []
        actual_actions = []
        actual_rewards = []
        
        for _, row in test_df.iterrows():
            state = self.extract_state_from_history(row)
            probs, _, _ = self._forward_actor(state)
            predicted_action = np.argmax(probs[0])
            actual_action = self.extract_action_from_history(row)
            actual_reward = self.extract_reward(row)
            
            predictions.append(predicted_action)
            actual_actions.append(actual_action)
            actual_rewards.append(actual_reward)
        
        accuracy = np.mean(np.array(predictions) == np.array(actual_actions))
        avg_actual_reward = np.mean(actual_rewards)
        
        return {
            'action_accuracy': accuracy,
            'avg_historical_reward': avg_actual_reward,
            'test_samples': len(test_df)
        }
    
    def predict_best_action(self, state: np.ndarray) -> Tuple[int, np.ndarray]:
        """预测最佳动作"""
        if state.ndim == 1:
            state = state.reshape(1, -1)
        probs, _, _ = self._forward_actor(state)
        best_action = np.argmax(probs[0])
        return best_action, probs[0]
    
    def get_action_probabilities(self, state: np.ndarray) -> np.ndarray:
        """获取所有动作的概率分布"""
        if state.ndim == 1:
            state = state.reshape(1, -1)
        probs, _, _ = self._forward_actor(state)
        return probs[0]


# ================== 主程序 ==================
def main_offline_training():
    """演示完整的离线训练流程"""
    print("="*70)
    print("基于历史数据的离线强化学习训练系统（完全修复版）")
    print("="*70)
    
    # 1. 构建历史数据集
    builder = OfflineDatasetBuilder()
    
    print("\n[步骤1] 构建历史数据集...")
    historical_df = builder.generate_synthetic_history(n_samples=10000)  # 减少样本数加快测试
    print(f"✓ 生成 {len(historical_df)} 条历史记录")
    print(f"  涉及市场: {historical_df['market'].unique()}")
    
    # 2. 数据预处理
    print("\n[步骤2] 数据预处理...")
    historical_df = historical_df[historical_df['conversion_rate'] <= 0.95]
    historical_df = historical_df[historical_df['price_usd'] <= 200]
    historical_df = historical_df.sort_values('timestamp')
    
    # 划分训练集和测试集
    split_idx = int(len(historical_df) * 0.8)
    train_df = historical_df.iloc[:split_idx]
    test_df = historical_df.iloc[split_idx:]
    
    print(f"✓ 训练集: {len(train_df)} 条")
    print(f"✓ 测试集: {len(test_df)} 条")
    
    # 3. 初始化离线训练器
    print("\n[步骤3] 初始化离线RL训练器...")
    trainer = OfflineRLTrainer(state_dim=5, action_dim=5, hidden_dim=32)
    print("✓ 模型初始化完成")
    
    # 4. 批量训练
    print("\n[步骤4] 开始批量训练...")
    loss_history = trainer.train_from_batch(train_df, epochs=20, batch_size=128, verbose=True)
    
    # 5. 模型评估
    print("\n[步骤5] 模型评估...")
    eval_metrics = trainer.evaluate_on_holdout(test_df)
    
    print(f"\n📊 评估结果:")
    print(f"  动作预测准确率: {eval_metrics['action_accuracy']:.2%}")
    print(f"  历史平均奖励: {eval_metrics['avg_historical_reward']:.3f}")
    print(f"  测试样本数: {eval_metrics['test_samples']}")
    
    # 6. 演示预测
    print("\n[步骤6] 策略预测演示...")
    sample_state = np.array([0.65, 0.55, 0.35, 0.18, 0.75])
    best_action, probs = trainer.predict_best_action(sample_state)
    
    price_map = {0: "低价策略 ($20-30)", 1: "中低价 ($30-50)", 
                 2: "中等价 ($50-70)", 3: "中高价 ($70-100)", 
                 4: "高端价 ($100+)"}
    
    print(f"\n  当前市场状态:")
    print(f"    品牌力: {sample_state[0]:.2f}")
    print(f"    数字化程度: {sample_state[1]:.2f}")
    print(f"    文化距离: {sample_state[2]:.2f}")
    print(f"    关税风险: {sample_state[3]:.2f}")
    print(f"    物流可达性: {sample_state[4]:.2f}")
    
    print(f"\n  推荐策略: {price_map[best_action]}")
    print(f"  各策略概率: {probs}")
    
    # 7. 保存模型
    print("\n[步骤7] 保存模型...")
    import pickle
    with open('offline_rl_model.pkl', 'wb') as f:
        pickle.dump(trainer, f)
    print("✓ 模型已保存到 offline_rl_model.pkl")
    
    return trainer, eval_metrics


if __name__ == "__main__":
    # 运行离线训练
    model, metrics = main_offline_training()
    
    print("\n" + "="*70)
    print("✅ 训练完成！模型可用于策略推荐")
    print("="*70)
    
    # 演示使用示例
    print("\n📝 使用示例:")
    print("""
# 加载模型
import pickle
with open('offline_rl_model.pkl', 'rb') as f:
    model = pickle.load(f)

# 为新策略做预测
current_state = np.array([0.7, 0.6, 0.4, 0.2, 0.8])
best_action, probs = model.predict_best_action(current_state)

# 根据动作执行策略
price_strategies = {0: 25, 1: 45, 2: 65, 3: 90, 4: 120}
recommended_price = price_strategies[best_action]
print(f"推荐定价: ${recommended_price}")

# 获取所有动作的概率
all_probs = model.get_action_probabilities(current_state)
print(f"各定价策略概率: {all_probs}")
""")

基于历史数据的离线强化学习训练系统（完全修复版）

[步骤1] 构建历史数据集...
✓ 生成 10000 条历史记录
  涉及市场: ['Southeast_Asia' 'North_America' 'Latin_America' 'Middle_East' 'Europe']

[步骤2] 数据预处理...
✓ 训练集: 8000 条
✓ 测试集: 2000 条

[步骤3] 初始化离线RL训练器...
✓ 模型初始化完成

[步骤4] 开始批量训练...

离线强化学习训练开始
历史数据量: 8000 条记录
训练轮数: 20, 批次大小: 128

Epoch 5/20 - Actor Loss: -1.8914, Critic Loss: 2.3071, Total: -0.7379
Epoch 10/20 - Actor Loss: -3.8715, Critic Loss: 6.4653, Total: -0.6389
Epoch 15/20 - Actor Loss: nan, Critic Loss: nan, Total: nan
Epoch 20/20 - Actor Loss: nan, Critic Loss: nan, Total: nan

✓ 训练完成！最终损失: nan

[步骤5] 模型评估...

📊 评估结果:
  动作预测准确率: 7.20%
  历史平均奖励: -0.648
  测试样本数: 2000

[步骤6] 策略预测演示...

  当前市场状态:
    品牌力: 0.65
    数字化程度: 0.55
    文化距离: 0.35
    关税风险: 0.18
    物流可达性: 0.75

  推荐策略: 低价策略 ($20-30)
  各策略概率: [nan nan nan nan nan]

[步骤7] 保存模型...
✓ 模型已保存到 offline_rl_model.pkl

✅ 训练完成！模型可用于策略推荐

📝 使用示例:

# 加载模型
import pickle
with open('offline_rl_model.pkl', 'rb') as f:
    model = pickle.load(f)

# 为新策略做预测
current_state = np.array([0.7,

In [7]:
# 加载模型
import pickle
with open('stable_offline_rl_model.pkl', 'rb') as f:
    model = pickle.load(f)

# 为新策略做预测
current_state = np.array([0.7, 0.6, 0.4, 0.2, 0.8])
best_action, probs = model.predict_best_action(current_state)

# 根据动作执行策略
price_strategies = {0: 25, 1: 45, 2: 65, 3: 90, 4: 120}
recommended_price = price_strategies[best_action]
print(f"推荐定价: ${recommended_price}")

# 获取所有动作的概率
all_probs = model.get_action_probabilities(current_state)
print(f"各定价策略概率: {all_probs}")

推荐定价: $120
各定价策略概率: [0.19853767 0.19933923 0.19999183 0.20017133 0.20195994]


In [6]:
import numpy as np
import pandas as pd
from collections import defaultdict
from datetime import datetime, timedelta
import random
from typing import List, Dict, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

# ================== 离线数据集构建器 ==================
class OfflineDatasetBuilder:
    """从历史数据构建离线训练数据集"""
    def __init__(self):
        self.dataset = []
        
    def build_from_csv(self, filepath: str) -> pd.DataFrame:
        df = pd.read_csv(filepath)
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        return df
    
    def generate_synthetic_history(self, n_samples: int = 10000) -> pd.DataFrame:
        """生成模拟历史数据（改进版，数值更稳定）"""
        np.random.seed(42)
        
        markets = ['Middle_East', 'Southeast_Asia', 'Europe', 'North_America', 'Latin_America']
        categories = ['decor', 'textile', 'ceramic', 'jewelry', 'furniture']
        
        data = {
            'timestamp': [datetime.now() - timedelta(days=np.random.randint(0, 365)) 
                         for _ in range(n_samples)],
            'market': np.random.choice(markets, n_samples),
            'product_category': np.random.choice(categories, n_samples),
            'price_usd': np.random.uniform(20, 150, n_samples),
            'marketing_budget_pct': np.random.uniform(0, 80, n_samples),
            'logistics_cost': np.random.uniform(100, 600, n_samples),
            'tariff_rate': np.random.uniform(0.05, 0.30, n_samples),
            'brand_power_before': np.random.uniform(0.3, 0.9, n_samples),
            'cultural_distance': np.random.uniform(0.2, 0.8, n_samples),
            'digital_maturity': np.random.uniform(0.3, 0.9, n_samples),
            'logistics_accessibility': np.random.uniform(0.4, 0.95, n_samples),
        }
        
        df = pd.DataFrame(data)
        
        # 模拟转化率（限制范围，避免极端值）
        df['conversion_rate'] = (
            0.3 * (1 - np.clip(df['price_usd'] / 200, 0, 1)) +
            0.2 * (df['marketing_budget_pct'] / 100) +
            0.15 * df['brand_power_before'] +
            0.1 * (1 - df['cultural_distance']) +
            0.1 * (1 / (1 + df['tariff_rate'] * 2)) +
            np.random.normal(0, 0.02, n_samples)  # 减小噪声
        )
        df['conversion_rate'] = df['conversion_rate'].clip(0.1, 0.9)
        
        # 模拟品牌力变化
        df['brand_power_after'] = df['brand_power_before'] + 0.05 * df['conversion_rate']
        df['brand_power_after'] = df['brand_power_after'].clip(0.1, 1.0)
        
        # 模拟奖励分数（归一化到[0,1]范围）
        profit_margin = (df['price_usd'] - df['logistics_cost']) / df['price_usd']
        profit_margin = profit_margin.clip(0, 0.5)  # 限制利润率范围
        
        df['reward'] = (
            0.5 * df['conversion_rate'] +
            0.3 * (df['brand_power_after'] - df['brand_power_before']).clip(0, 0.3) +
            0.2 * profit_margin
        )
        df['reward'] = df['reward'].clip(0.1, 0.9)  # 奖励限制在合理范围
        
        return df


# ================== 稳定的离线强化学习训练器 ==================
class StableOfflineRLTrainer:
    """稳定的离线强化学习训练器（解决NaN问题）"""
    
    def __init__(self, state_dim: int = 5, action_dim: int = 5, hidden_dim: int = 32):
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.hidden_dim = hidden_dim
        
        # 使用更小的初始化范围
        init_scale = 0.01
        
        # 初始化Actor网络权重
        self.actor_W1 = np.random.randn(state_dim, hidden_dim) * init_scale
        self.actor_b1 = np.zeros((1, hidden_dim))
        self.actor_W2 = np.random.randn(hidden_dim, action_dim) * init_scale
        self.actor_b2 = np.zeros((1, action_dim))
        
        # 初始化Critic网络权重
        self.critic_W1 = np.random.randn(state_dim, hidden_dim) * init_scale
        self.critic_b1 = np.zeros((1, hidden_dim))
        self.critic_W2 = np.random.randn(hidden_dim, 1) * init_scale
        self.critic_b2 = np.zeros((1, 1))
        
        # 更小的学习率
        self.learning_rate = 0.0001
        self.gamma = 0.95
        
        # 梯度裁剪阈值
        self.grad_clip = 1.0
        
    def _relu(self, x):
        return np.maximum(0, x)
    
    def _softmax(self, x):
        """数值稳定的softmax"""
        x = x - np.max(x, axis=-1, keepdims=True)
        exp_x = np.exp(np.clip(x, -100, 100))  # 防止指数爆炸
        return exp_x / (np.sum(exp_x, axis=-1, keepdims=True) + 1e-8)
    
    def _forward_actor(self, state):
        """策略网络前向传播"""
        if state.ndim == 1:
            state = state.reshape(1, -1)
        
        # 第一层
        hidden = self._relu(np.dot(state, self.actor_W1) + self.actor_b1)
        # 第二层
        logits = np.dot(hidden, self.actor_W2) + self.actor_b2
        # Softmax
        probs = self._softmax(logits)
        
        return probs, logits, hidden
    
    def _forward_critic(self, state):
        """价值网络前向传播"""
        if state.ndim == 1:
            state = state.reshape(1, -1)
        
        # 第一层
        hidden = self._relu(np.dot(state, self.critic_W1) + self.critic_b1)
        # 第二层
        value = np.dot(hidden, self.critic_W2) + self.critic_b2
        
        return value[0, 0], hidden
    
    def extract_state_from_history(self, row: pd.Series) -> np.ndarray:
        """从历史数据提取状态向量（归一化）"""
        return np.array([
            float(row.get('brand_power_before', 0.5)),
            float(row.get('digital_maturity', 0.5)),
            float(row.get('cultural_distance', 0.5)),
            float(row.get('tariff_rate', 0.15)),
            float(row.get('logistics_accessibility', 0.7))
        ]).clip(0, 1)  # 确保在[0,1]范围
    
    def extract_action_from_history(self, row: pd.Series) -> int:
        """从历史数据提取动作"""
        price = row.get('price_usd', 50)
        if price < 30:
            return 0
        elif price < 50:
            return 1
        elif price < 70:
            return 2
        elif price < 100:
            return 3
        else:
            return 4
    
    def extract_reward(self, row: pd.Series) -> float:
        """提取或计算奖励值（确保在稳定范围）"""
        if 'reward' in row:
            reward = float(row['reward'])
        else:
            conversion = float(row.get('conversion_rate', 0.3))
            price = float(row.get('price_usd', 50))
            logistics = float(row.get('logistics_cost', 30))
            profit_margin = (price - logistics) / price if price > 0 else 0
            reward = 0.6 * conversion + 0.4 * max(0, profit_margin)
        
        # 裁剪奖励范围，避免极端值
        return np.clip(reward, 0.1, 0.9)
    
    def clip_gradients(self, gradients):
        """梯度裁剪，防止梯度爆炸"""
        for key in gradients:
            gradients[key] = np.clip(gradients[key], -self.grad_clip, self.grad_clip)
        return gradients
    
    def compute_actor_gradients(self, state, action, advantage):
        """计算Actor网络梯度（稳定版）"""
        if state.ndim == 1:
            state = state.reshape(1, -1)
        
        # 裁剪优势函数
        advantage = np.clip(advantage, -10, 10)
        
        # 前向传播
        hidden = self._relu(np.dot(state, self.actor_W1) + self.actor_b1)
        logits = np.dot(hidden, self.actor_W2) + self.actor_b2
        probs = self._softmax(logits)
        
        # 添加小常数防止log(0)
        probs = np.clip(probs, 1e-8, 1.0)
        
        # 计算策略梯度
        grad_logits = probs.copy()
        grad_logits[0, action] -= 1
        
        # 第二层梯度
        grad_W2 = np.dot(hidden.T, grad_logits)
        grad_b2 = np.sum(grad_logits, axis=0, keepdims=True)
        
        # 第一层梯度
        grad_hidden = np.dot(grad_logits, self.actor_W2.T)
        relu_mask = (hidden > 0).astype(float)
        grad_hidden = grad_hidden * relu_mask
        
        grad_W1 = np.dot(state.T, grad_hidden)
        grad_b1 = np.sum(grad_hidden, axis=0, keepdims=True)
        
        # 应用优势函数
        grad_W1 *= advantage
        grad_b1 *= advantage
        grad_W2 *= advantage
        grad_b2 *= advantage
        
        gradients = {
            'W1': grad_W1,
            'b1': grad_b1,
            'W2': grad_W2,
            'b2': grad_b2
        }
        
        return self.clip_gradients(gradients)
    
    def compute_critic_gradients(self, state, td_error):
        """计算Critic网络梯度（稳定版）"""
        if state.ndim == 1:
            state = state.reshape(1, -1)
        
        # 裁剪TD误差
        td_error = np.clip(td_error, -10, 10)
        
        # 前向传播
        hidden = self._relu(np.dot(state, self.critic_W1) + self.critic_b1)
        value = np.dot(hidden, self.critic_W2) + self.critic_b2
        
        # 价值损失梯度
        grad_value = td_error
        
        # 第二层梯度
        grad_W2 = np.dot(hidden.T, grad_value)
        grad_b2 = np.sum(grad_value, axis=0, keepdims=True)
        
        # 第一层梯度
        grad_hidden = np.dot(grad_value, self.critic_W2.T)
        relu_mask = (hidden > 0).astype(float)
        grad_hidden = grad_hidden * relu_mask
        
        grad_W1 = np.dot(state.T, grad_hidden)
        grad_b1 = np.sum(grad_hidden, axis=0, keepdims=True)
        
        gradients = {
            'W1': grad_W1,
            'b1': grad_b1,
            'W2': grad_W2,
            'b2': grad_b2
        }
        
        return self.clip_gradients(gradients)
    
    def update_network(self, gradients, network_type='actor'):
        """更新网络权重（使用更小的学习率）"""
        lr = self.learning_rate
        
        if network_type == 'actor':
            self.actor_W1 -= lr * gradients['W1']
            self.actor_b1 -= lr * gradients['b1']
            self.actor_W2 -= lr * gradients['W2']
            self.actor_b2 -= lr * gradients['b2']
        else:  # critic
            self.critic_W1 -= lr * gradients['W1']
            self.critic_b1 -= lr * gradients['b1']
            self.critic_W2 -= lr * gradients['W2']
            self.critic_b2 -= lr * gradients['b2']
    
    def train_from_batch(self, historical_df: pd.DataFrame, epochs: int = 20, 
                        batch_size: int = 128, verbose: bool = True):
        """从历史数据批量训练（稳定版）"""
        print(f"\n{'='*60}")
        print(f"离线强化学习训练开始")
        print(f"历史数据量: {len(historical_df)} 条记录")
        print(f"训练轮数: {epochs}, 批次大小: {batch_size}")
        print(f"学习率: {self.learning_rate}, 梯度裁剪: {self.grad_clip}")
        print(f"{'='*60}\n")
        
        loss_history = []
        
        for epoch in range(epochs):
            # 随机采样批次
            batch = historical_df.sample(n=min(batch_size, len(historical_df)))
            
            epoch_actor_losses = []
            epoch_critic_losses = []
            
            for _, row in batch.iterrows():
                try:
                    # 提取数据
                    state = self.extract_state_from_history(row)
                    action = self.extract_action_from_history(row)
                    reward = self.extract_reward(row)
                    
                    # 计算当前价值
                    current_value, _ = self._forward_critic(state)
                    
                    # TD误差
                    td_error = reward - current_value
                    
                    # 计算梯度
                    actor_grads = self.compute_actor_gradients(state, action, td_error)
                    critic_grads = self.compute_critic_gradients(state, td_error)
                    
                    # 更新网络
                    self.update_network(actor_grads, 'actor')
                    self.update_network(critic_grads, 'critic')
                    
                    # 计算损失（用于监控）
                    probs, _, _ = self._forward_actor(state)
                    probs = np.clip(probs, 1e-8, 1.0)
                    log_prob = np.log(probs[0, action])
                    policy_loss = -log_prob * td_error
                    value_loss = td_error ** 2
                    
                    # 检查数值有效性
                    if not np.isnan(policy_loss) and not np.isinf(policy_loss):
                        epoch_actor_losses.append(policy_loss)
                    if not np.isnan(value_loss) and not np.isinf(value_loss):
                        epoch_critic_losses.append(value_loss)
                        
                except Exception as e:
                    continue
            
            # 计算平均损失
            if epoch_actor_losses and epoch_critic_losses:
                avg_actor_loss = np.mean(epoch_actor_losses)
                avg_critic_loss = np.mean(epoch_critic_losses)
                total_loss = avg_actor_loss + 0.5 * avg_critic_loss
                loss_history.append(total_loss)
            else:
                loss_history.append(1.0)
                avg_actor_loss = 0
                avg_critic_loss = 0
                total_loss = 1.0
            
            if verbose and (epoch + 1) % 5 == 0:
                print(f"Epoch {epoch+1}/{epochs} - "
                      f"Actor Loss: {avg_actor_loss:.4f}, "
                      f"Critic Loss: {avg_critic_loss:.4f}, "
                      f"Total: {total_loss:.4f}")
        
        print(f"\n✓ 训练完成！最终损失: {loss_history[-1]:.4f}")
        return loss_history
    
    def evaluate_on_holdout(self, test_df: pd.DataFrame) -> Dict:
        """评估模型性能"""
        predictions = []
        actual_actions = []
        actual_rewards = []
        
        for _, row in test_df.iterrows():
            try:
                state = self.extract_state_from_history(row)
                probs, _, _ = self._forward_actor(state)
                predicted_action = np.argmax(probs[0])
                actual_action = self.extract_action_from_history(row)
                actual_reward = self.extract_reward(row)
                
                predictions.append(predicted_action)
                actual_actions.append(actual_action)
                actual_rewards.append(actual_reward)
            except:
                continue
        
        if len(predictions) == 0:
            return {
                'action_accuracy': 0.0,
                'avg_historical_reward': 0.0,
                'test_samples': 0
            }
        
        accuracy = np.mean(np.array(predictions) == np.array(actual_actions))
        avg_actual_reward = np.mean(actual_rewards)
        
        return {
            'action_accuracy': accuracy,
            'avg_historical_reward': avg_actual_reward,
            'test_samples': len(predictions)
        }
    
    def predict_best_action(self, state: np.ndarray) -> Tuple[int, np.ndarray]:
        """预测最佳动作"""
        if state.ndim == 1:
            state = state.reshape(1, -1)
        probs, _, _ = self._forward_actor(state)
        best_action = np.argmax(probs[0])
        return best_action, probs[0]
    
    def get_action_probabilities(self, state: np.ndarray) -> np.ndarray:
        """获取所有动作的概率分布"""
        if state.ndim == 1:
            state = state.reshape(1, -1)
        probs, _, _ = self._forward_actor(state)
        return probs[0]


# ================== 主程序 ==================
def main_offline_training():
    """演示完整的离线训练流程"""
    print("="*70)
    print("稳定的离线强化学习训练系统（解决NaN问题）")
    print("="*70)
    
    # 1. 构建历史数据集
    builder = OfflineDatasetBuilder()
    
    print("\n[步骤1] 构建历史数据集...")
    historical_df = builder.generate_synthetic_history(n_samples=8000)
    print(f"✓ 生成 {len(historical_df)} 条历史记录")
    print(f"  涉及市场: {historical_df['market'].unique()}")
    
    # 2. 数据预处理
    print("\n[步骤2] 数据预处理...")
    historical_df = historical_df[historical_df['conversion_rate'] <= 0.95]
    historical_df = historical_df[historical_df['price_usd'] <= 200]
    historical_df = historical_df.sort_values('timestamp')
    
    # 划分训练集和测试集
    split_idx = int(len(historical_df) * 0.8)
    train_df = historical_df.iloc[:split_idx]
    test_df = historical_df.iloc[split_idx:]
    
    print(f"✓ 训练集: {len(train_df)} 条")
    print(f"✓ 测试集: {len(test_df)} 条")
    
    # 3. 初始化稳定的训练器
    print("\n[步骤3] 初始化稳定版离线RL训练器...")
    trainer = StableOfflineRLTrainer(state_dim=5, action_dim=5, hidden_dim=32)
    print("✓ 模型初始化完成")
    print("  改进点：更小的学习率(0.0001)、梯度裁剪、数值稳定的softmax")
    
    # 4. 批量训练
    print("\n[步骤4] 开始批量训练...")
    loss_history = trainer.train_from_batch(train_df, epochs=20, batch_size=128, verbose=True)
    
    # 5. 模型评估
    print("\n[步骤5] 模型评估...")
    eval_metrics = trainer.evaluate_on_holdout(test_df)
    
    print(f"\n📊 评估结果:")
    print(f"  动作预测准确率: {eval_metrics['action_accuracy']:.2%}")
    print(f"  历史平均奖励: {eval_metrics['avg_historical_reward']:.3f}")
    print(f"  测试样本数: {eval_metrics['test_samples']}")
    
    # 6. 演示预测
    print("\n[步骤6] 策略预测演示...")
    sample_state = np.array([0.65, 0.55, 0.35, 0.18, 0.75])
    best_action, probs = trainer.predict_best_action(sample_state)
    
    price_map = {0: "低价策略 ($20-30)", 1: "中低价 ($30-50)", 
                 2: "中等价 ($50-70)", 3: "中高价 ($70-100)", 
                 4: "高端价 ($100+)"}
    
    print(f"\n  当前市场状态:")
    print(f"    品牌力: {sample_state[0]:.2f}")
    print(f"    数字化程度: {sample_state[1]:.2f}")
    print(f"    文化距离: {sample_state[2]:.2f}")
    print(f"    关税风险: {sample_state[3]:.2f}")
    print(f"    物流可达性: {sample_state[4]:.2f}")
    
    print(f"\n  推荐策略: {price_map[best_action]}")
    print(f"  各策略概率: {probs}")
    
    # 7. 保存模型
    print("\n[步骤7] 保存模型...")
    import pickle
    with open('stable_offline_rl_model.pkl', 'wb') as f:
        pickle.dump(trainer, f)
    print("✓ 模型已保存到 stable_offline_rl_model.pkl")
    
    return trainer, eval_metrics


if __name__ == "__main__":
    # 运行离线训练
    model, metrics = main_offline_training()
    
    print("\n" + "="*70)
    print("✅ 训练完成！模型可用于策略推荐")
    print("="*70)

稳定的离线强化学习训练系统（解决NaN问题）

[步骤1] 构建历史数据集...
✓ 生成 8000 条历史记录
  涉及市场: ['Southeast_Asia' 'Middle_East' 'Latin_America' 'Europe' 'North_America']

[步骤2] 数据预处理...
✓ 训练集: 6400 条
✓ 测试集: 1600 条

[步骤3] 初始化稳定版离线RL训练器...
✓ 模型初始化完成
  改进点：更小的学习率(0.0001)、梯度裁剪、数值稳定的softmax

[步骤4] 开始批量训练...

离线强化学习训练开始
历史数据量: 6400 条记录
训练轮数: 20, 批次大小: 128
学习率: 0.0001, 梯度裁剪: 1.0

Epoch 5/20 - Actor Loss: 0.4072, Critic Loss: 0.0658, Total: 0.4401
Epoch 10/20 - Actor Loss: 0.4337, Critic Loss: 0.0745, Total: 0.4709
Epoch 15/20 - Actor Loss: 0.4761, Critic Loss: 0.0899, Total: 0.5210
Epoch 20/20 - Actor Loss: 0.5063, Critic Loss: 0.1009, Total: 0.5568

✓ 训练完成！最终损失: 0.5568

[步骤5] 模型评估...

📊 评估结果:
  动作预测准确率: 39.69%
  历史平均奖励: 0.242
  测试样本数: 1600

[步骤6] 策略预测演示...

  当前市场状态:
    品牌力: 0.65
    数字化程度: 0.55
    文化距离: 0.35
    关税风险: 0.18
    物流可达性: 0.75

  推荐策略: 高端价 ($100+)
  各策略概率: [0.19853134 0.19934346 0.19998149 0.20018285 0.20196086]

[步骤7] 保存模型...
✓ 模型已保存到 stable_offline_rl_model.pkl

✅ 训练完成！模型可用于策略推荐
